In [ ]:
!pip install mne

In [ ]:
import os
import mne
import pandas as pd
import numpy as np
from collections import Counter

In [ ]:

# ============================================================
# TASK 1 - PART A1: EEG DATA AUDIT
# Dataset: PhysioNet EEG Motor Movement/Imagery (EEGBCI)
# Subjects: 70-109 (40 subjects)
# Runs: 4, 8, 12 (left vs right fist motor imagery)
# Output: audit.csv
# ============================================================

# ------------------------------------------------------------
# 1. EXPERIMENT CONFIGURATION
# ------------------------------------------------------------

SUBJECTS = list(range(70, 110))

RUNS = [4, 8, 12]

DATA_DIR = "/content/mne_data"

OUTPUT_FILE = "/content/audit.csv"

EXPECTED_SAMPLING_RATE = 160.0

os.makedirs(DATA_DIR, exist_ok=True)

# Event-code interpretation for selected runs
EVENT_MAPPING = {
    "T0": "Rest (excluded)",
    "T1": "Left fist imagery",
    "T2": "Right fist imagery"
}

print("TASK 1 - MOTOR IMAGERY DATA AUDIT")
print("---------------------------------")
print("Total subjects:", len(SUBJECTS))
print("Subjects:", SUBJECTS[0], "to", SUBJECTS[-1])
print("Selected runs:", RUNS)


# ------------------------------------------------------------
# 2. FUNCTION TO AUDIT ONE SUBJECT
# ------------------------------------------------------------

def audit_subject(subject):

    loaded_runs = []
    failed_runs = []

    sampling_rates = []

    total_left = 0
    total_right = 0

    anomalies = []

    run_details = []

    # Process each selected run separately
    for run in RUNS:

        try:

            print(f"Subject {subject} | Loading run {run}")

            # Download EEG recording from PhysioNet
            file_paths = mne.datasets.eegbci.load_data(
                subject,
                [run],
                path=DATA_DIR,
                verbose="ERROR"
            )

            if not file_paths:
                raise RuntimeError("No recording path returned")

            # Read EDF recording
            raw = mne.io.read_raw_edf(
                file_paths[0],
                preload=False,
                verbose="ERROR"
            )

            loaded_runs.append(run)

            # Obtain sampling frequency
            sfreq = float(raw.info["sfreq"])

            sampling_rates.append(sfreq)

            if not np.isclose(
                sfreq,
                EXPECTED_SAMPLING_RATE
            ):
                anomalies.append(
                    f"Run {run}: unexpected sampling rate {sfreq} Hz"
                )

            # Read annotation descriptions
            descriptions = [
                str(x) for x in raw.annotations.description
            ]

            event_counts = Counter(descriptions)

            # Count left and right imagery trials
            left_count = event_counts.get("T1", 0)

            right_count = event_counts.get("T2", 0)

            rest_count = event_counts.get("T0", 0)

            total_left += left_count

            total_right += right_count

            # Check whether expected classes are present
            if left_count == 0:
                anomalies.append(
                    f"Run {run}: no T1 (left imagery) events"
                )

            if right_count == 0:
                anomalies.append(
                    f"Run {run}: no T2 (right imagery) events"
                )

            # Check for unexpected annotation labels
            unexpected = sorted(
                set(descriptions) - {"T0", "T1", "T2"}
            )

            if unexpected:
                anomalies.append(
                    f"Run {run}: unexpected annotations {unexpected}"
                )

            # Record run-level details
            run_details.append(
                f"Run {run}: T0={rest_count}, "
                f"T1={left_count}, T2={right_count}"
            )

            raw.close()

        except Exception as e:

            failed_runs.append(run)

            anomalies.append(
                f"Run {run}: loading/reading failed - "
                f"{type(e).__name__}: {str(e)}"
            )

    # Check for missing runs
    if failed_runs:

        anomalies.append(
            f"Failed runs: {failed_runs}"
        )

    # Sampling rate summary
    if sampling_rates:

        unique_rates = sorted(set(sampling_rates))

        sampling_rate = ", ".join(
            str(x) for x in unique_rates
        )

        if len(unique_rates) > 1:

            anomalies.append(
                "Sampling rates differ across runs"
            )

    else:

        sampling_rate = "Unavailable"

    # Record subject-level audit
    result = {

        "subject_id": subject,

        "runs_loaded": ",".join(
            str(x) for x in loaded_runs
        ),

        "runs_failed": ",".join(
            str(x) for x in failed_runs
        ),

        "sampling_rate_hz": sampling_rate,

        "left_fist_trials_T1": total_left,

        "right_fist_trials_T2": total_right,

        "total_imagery_trials": total_left + total_right,

        "run_event_counts": " | ".join(run_details),

        "event_code_meaning": (
            "T0=Rest (excluded); "
            "T1=Left fist imagery; "
            "T2=Right fist imagery"
        ),

        "anomalies": (
            "; ".join(anomalies)
            if anomalies
            else "None"
        )
    }

    return result


# ------------------------------------------------------------
# 3. AUDIT ALL 40 SUBJECTS
# ------------------------------------------------------------

audit_results = []

for subject in SUBJECTS:

    print("\n===================================")
    print("Auditing subject:", subject)
    print("===================================")

    subject_result = audit_subject(subject)

    audit_results.append(subject_result)


# ------------------------------------------------------------
# 4. CREATE AUDIT DATAFRAME
# ------------------------------------------------------------

audit_df = pd.DataFrame(audit_results)


# ------------------------------------------------------------
# 5. SAVE AUDIT CSV
# ------------------------------------------------------------

audit_df.to_csv(
    OUTPUT_FILE,
    index=False
)


# ------------------------------------------------------------
# 6. DISPLAY RESULTS
# ------------------------------------------------------------

print("\n\n========== AUDIT COMPLETED ==========")

print("\nTotal subjects audited:",
      len(audit_df))

print("\nSuccessfully loaded all three runs:",
      sum(
          audit_df["runs_loaded"].apply(
              lambda x: set(x.split(",")) == {"4", "8", "12"}
              if x else False
          )
      ))

print("\nSubjects with anomalies:",
      sum(audit_df["anomalies"] != "None"))

print("\nTotal left imagery trials:",
      audit_df["left_fist_trials_T1"].sum())

print("\nTotal right imagery trials:",
      audit_df["right_fist_trials_T2"].sum())

print("\nAudit table:")

display(audit_df)

print("\nCSV saved successfully at:")
print(OUTPUT_FILE)


TASK 1 - MOTOR IMAGERY DATA AUDIT
---------------------------------
Total subjects: 40
Subjects: 70 to 109
Selected runs: [4, 8, 12]

Auditing subject: 70
Subject 70 | Loading run 4


Do you want to set the path:
    /content/mne_data
as the default EEGBCI dataset path in the mne-python config [y]/n? y


Subject 70 | Loading run 8


Subject 70 | Loading run 12



Auditing subject: 71
Subject 71 | Loading run 4


Subject 71 | Loading run 8
Subject 71 | Loading run 12


... (108 outputs left out)

Subject 109 | Loading run 8


Subject 109 | Loading run 12


========== AUDIT COMPLETED ==========

Total subjects audited: 40

Successfully loaded all three runs: 40

Subjects with anomalies: 3

Total left imagery trials: 909

Total right imagery trials: 904

Audit table:


,subject_id,runs_loaded,runs_failed,sampling_rate_hz,left_fist_trials_T1,right_fist_trials_T2,total_imagery_trials,run_event_counts,event_code_meaning,anomalies
0,70,"4,8,12",,160.0,22,23,45,"Run 4: T0=15, T1=7, T2=8 | Run 8: T0=15, T1=8,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None
1,71,"4,8,12",,160.0,22,23,45,"Run 4: T0=15, T1=8, T2=7 | Run 8: T0=15, T1=7,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None
2,72,"4,8,12",,160.0,22,23,45,"Run 4: T0=15, T1=7, T2=8 | Run 8: T0=15, T1=7,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None
3,73,"4,8,12",,160.0,23,22,45,"Run 4: T0=15, T1=8, T2=7 | Run 8: T0=15, T1=8,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None
4,74,"4,8,12",,160.0,24,21,45,"Run 4: T0=15, T1=8, T2=7 | Run 8: T0=15, T1=8,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None
5,75,"4,8,12",,160.0,22,23,45,"Run 4: T0=15, T1=7, T2=8 | Run 8: T0=15, T1=7,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None
6,76,"4,8,12",,160.0,23,22,45,"Run 4: T0=15, T1=8, T2=7 | Run 8: T0=15, T1=8,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None
7,77,"4,8,12",,160.0,21,24,45,"Run 4: T0=15, T1=7, T2=8 | Run 8: T0=15, T1=7,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None
8,78,"4,8,12",,160.0,23,22,45,"Run 4: T0=15, T1=8, T2=7 | Run 8: T0=15, T1=7,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None
9,79,"4,8,12",,160.0,22,23,45,"Run 4: T0=15, T1=8, T2=7 | Run 8: T0=15, T1=7,...",T0=Rest (excluded); T1=Left fist imagery; T2=R...,None



CSV saved successfully at:
/content/audit.csv


In [ ]:
from google.colab import files

files.download("/content/audit.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# ============================================================
# TASK 1 - A1: ADDITIONAL ANOMALY INVESTIGATION
# ============================================================

import mne
import pandas as pd
import numpy as np

SUBJECTS = range(70, 110)
RUNS = [4, 8, 12]

DATA_DIR = "/content/mne_data"

diagnostic_results = []

for subject in SUBJECTS:

    for run in RUNS:

        try:

            # Get recording path
            paths = mne.datasets.eegbci.load_data(
                subject,
                [run],
                path=DATA_DIR,
                verbose="ERROR"
            )

            # Read recording
            raw = mne.io.read_raw_edf(
                paths[0],
                preload=False,
                verbose="ERROR"
            )

            # Sampling frequency
            sfreq = raw.info["sfreq"]

            # Recording duration
            duration = raw.n_times / sfreq

            # Number of recorded samples
            n_samples = raw.n_times

            # Annotation information
            descriptions = [
                str(x) for x in raw.annotations.description
            ]

            # Count event types
            t0 = descriptions.count("T0")
            t1 = descriptions.count("T1")
            t2 = descriptions.count("T2")

            total_events = len(descriptions)

            # Event onset information
            onset_times = raw.annotations.onset

            # Time interval between consecutive annotations
            if len(onset_times) > 1:
                intervals = np.diff(onset_times)
                median_interval = np.median(intervals)
            else:
                median_interval = np.nan

            # Flag unusual annotation counts
            issues = []

            # 30 is the modal annotation count in your audit.
            # This is a screening reference, not an exclusion rule.
            if total_events != 30:
                issues.append(
                    f"Non-modal annotation count: {total_events}"
                )

            if t0 != 15:
                issues.append(
                    f"Non-modal T0 count: {t0}"
                )

            if t1 + t2 != 15:
                issues.append(
                    f"Non-modal imagery event count: {t1+t2}"
                )

            if sfreq not in [128.0, 160.0]:
                issues.append(
                    f"Unexpected sampling rate: {sfreq}"
                )

            diagnostic_results.append({

                "subject_id": subject,
                "run": run,
                "sampling_rate_hz": sfreq,
                "duration_seconds": duration,
                "number_of_samples": n_samples,
                "T0_count": t0,
                "T1_count": t1,
                "T2_count": t2,
                "total_annotations": total_events,
                "median_annotation_interval": median_interval,
                "diagnostic_issues": (
                    "; ".join(issues)
                    if issues else "None"
                )

            })

            raw.close()

        except Exception as e:

            diagnostic_results.append({

                "subject_id": subject,
                "run": run,
                "sampling_rate_hz": np.nan,
                "duration_seconds": np.nan,
                "number_of_samples": np.nan,
                "T0_count": np.nan,
                "T1_count": np.nan,
                "T2_count": np.nan,
                "total_annotations": np.nan,
                "median_annotation_interval": np.nan,
                "diagnostic_issues": str(e)

            })


# Create detailed diagnostic table
diagnostic_df = pd.DataFrame(diagnostic_results)

# Save diagnostic results
diagnostic_df.to_csv(
    "/content/run_level_diagnostics.csv",
    index=False
)

# Display only unusual recordings
unusual = diagnostic_df[
    diagnostic_df["diagnostic_issues"] != "None"
]

print("Total recordings examined:", len(diagnostic_df))

print("\nUnusual recordings:")
display(unusual)

print("\nDiagnostic file saved:")
print("/content/run_level_diagnostics.csv")


Total recordings examined: 120

Unusual recordings:


,subject_id,run,sampling_rate_hz,duration_seconds,number_of_samples,T0_count,T1_count,T2_count,total_annotations,median_annotation_interval,diagnostic_issues
54,88,4,128.0,124.0,15872,19,10,9,38,1.400,Non-modal annotation count: 38; Non-modal T0 c...
55,88,8,128.0,124.0,15872,19,10,9,38,1.400,Non-modal annotation count: 38; Non-modal T0 c...
56,88,12,128.0,124.0,15872,19,9,10,38,1.400,Non-modal annotation count: 38; Non-modal T0 c...
66,92,4,128.0,124.0,15872,19,9,10,38,1.400,Non-modal annotation count: 38; Non-modal T0 c...
67,92,8,128.0,124.0,15872,19,9,10,38,1.400,Non-modal annotation count: 38; Non-modal T0 c...
68,92,12,128.0,124.0,15872,19,9,10,38,1.400,Non-modal annotation count: 38; Non-modal T0 c...
90,100,4,128.0,123.0,15744,12,6,6,24,5.125,Non-modal annotation count: 24; Non-modal T0 c...
91,100,8,128.0,123.0,15744,12,6,6,24,5.125,Non-modal annotation count: 24; Non-modal T0 c...
92,100,12,128.0,123.0,15744,12,6,6,24,5.125,Non-modal annotation count: 24; Non-modal T0 c...
103,104,8,160.0,106.0,16960,13,7,6,26,4.100,Non-modal annotation count: 26; Non-modal T0 c...



Diagnostic file saved:
/content/run_level_diagnostics.csv


In [ ]:
from google.colab import files

files.download("/content/run_level_diagnostics.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>